# V1 — Vector Diagnostics: structure & semantic similarity for ANY representation

A reusable battery. Point it at any hidden representation `X` (shape
`[n, d]`) — LLM pooled states, image embeddings, sentence vectors — and it
runs every structural test from the project. If you also have a **paired**
representation `Y` (row i = same input in another model), it runs the
cross-space battery too: CKA, Procrustes-vs-ridge, volume knobs, retrieval,
membership gap.

| Block | Tests | Needs |
|---|---|---|
| 1. Sanity | finite / zero rows / duplicates / norms / rows-per-dim | X |
| 2. Anisotropy | mean pairwise cosine, mean-direction ratio, PC1 & top-k share, rank-1 energy, effective rank, spectrum plot | X |
| 3. Robustness | centering + top-PC removal, before/after comparison | X |
| 4. Cross-space | held-out ridge R² + shuffle test, Procrustes vs ridge, CKA, singular values of W (volume knobs) | X, Y paired |
| 5. Semantic | R@K retrieval, membership gap (true-twin minus best impostor) | X, Y paired |

Every block prints calibrated reference points next to the measured value.

In [ ]:
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr

DATA_DIR = Path(os.environ["DATA_DIR"])
rng = np.random.default_rng(0)

# ---------- POINT THESE AT ANY REPRESENTATION ----------
X_FILE, X_KEY = "pairs.npz", "mob_img"      # required: [n, d1]
Y_FILE, Y_KEY = "pairs.npz", "sig_img"      # optional paired [n, d2]; None to skip
# Y_FILE = None

X = np.load(str(DATA_DIR / X_FILE))[X_KEY].astype(np.float64)
Y = (np.load(str(DATA_DIR / Y_FILE))[Y_KEY].astype(np.float64)
     if Y_FILE else None)
print("X", X.shape, "| Y", None if Y is None else Y.shape)

def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

## Block 1 — Sanity (run before believing anything)

In [ ]:
n, d = X.shape
print(f"rows/dim: {n}/{d} = {n/d:.1f}   (fitted maps need >=5)")
bad = int(np.isnan(X).any(1).sum() + np.isinf(X).any(1).sum())
norms = np.linalg.norm(X, axis=1)
nz = int((norms < 1e-9).sum())
samp = X[rng.choice(n, min(4000, n), replace=False)]
_, cnt = np.unique(np.round(samp, 5), axis=0, return_counts=True)
dup = int((cnt > 1).sum())
print(f"NaN/Inf rows {bad} | zero rows {nz} | duplicate rows {dup}")
print(f"norms: mean {norms.mean():.3f}  min {norms.min():.3f}  "
      f"max {norms.max():.3f}"
      + ("   <- unit-normalized" if abs(norms.mean()-1) < 1e-3 else ""))

## Block 2 — Anisotropy battery

| Statistic | isotropic looks like | concentrated looks like |
|---|---|---|
| mean random-pair cosine | ~0.00 | >0.3 |
| mean-direction ratio | ~0.0 | >0.5 |
| PC1 share | ~1/d | >0.3 |
| rank-1 energy | small | >0.5 |
| effective rank | ~d | << d |

In [ ]:
def anisotropy_report(M, name):
    Mn = l2n(M)
    k = min(2000, len(M))
    ii = rng.choice(len(M), k); jj = rng.choice(len(M), k)
    keep = ii != jj
    pair_cos = float((Mn[ii[keep]] * Mn[jj[keep]]).sum(1).mean())
    mdr = float(np.linalg.norm(M.mean(0)) /
                max(np.linalg.norm(M, axis=1).mean(), 1e-12))
    Xc = M - M.mean(0, keepdims=True)
    s = np.linalg.svd(Xc, full_matrices=False, compute_uv=False)
    e = s ** 2
    p = e / e.sum()
    pc1 = float(p[0]); topk = float(p[:10].sum())
    rank1 = pc1
    eff_rank = float(np.exp(-(p[p > 0] * np.log(p[p > 0])).sum()))
    pr = float(e.sum() ** 2 / (e ** 2).sum())     # participation ratio
    print(f"--- {name}  [n={M.shape[0]}, d={M.shape[1]}]")
    print(f"  mean random-pair cosine : {pair_cos:+.3f}   (isotropic ~0)")
    print(f"  mean-direction ratio    : {mdr:.3f}    (isotropic ~0)")
    print(f"  PC1 / top-10 var share  : {pc1:.3f} / {topk:.3f}")
    print(f"  rank-1 energy share     : {rank1:.3f}")
    print(f"  effective rank (entropy): {eff_rank:.1f} of {M.shape[1]}")
    print(f"  participation ratio     : {pr:.1f}")
    return s

sX = anisotropy_report(X, "X")
if Y is not None:
    sY = anisotropy_report(Y, "Y")

plt.figure(figsize=(7.5, 4))
plt.semilogy(sX ** 2 / (sX ** 2).sum(), label="X")
if Y is not None:
    plt.semilogy(sY ** 2 / (sY ** 2).sum(), label="Y")
plt.xlabel("principal direction"); plt.ylabel("variance share (log)")
plt.title("Spectrum: flat = isotropic, steep = concentrated")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(str(DATA_DIR / "v1_spectrum.png"), dpi=140); plt.show()

## Block 3 — Does structure survive the corrections?

Center, then remove top principal components. A finding that survives is
not an artifact of a shared dominant direction; one that vanishes was.

In [ ]:
def remove_top_pcs(M, k):
    Mc = M - M.mean(0, keepdims=True)
    U, s, Vt = np.linalg.svd(Mc, full_matrices=False)
    return Mc - (Mc @ Vt[:k].T) @ Vt[:k]

for k in (0, 1, 3, 10):
    Xk = remove_top_pcs(X, k) if k else X - X.mean(0, keepdims=True)
    Xn = l2n(Xk)
    kk = min(2000, len(X))
    ii = rng.choice(len(X), kk); jj = rng.choice(len(X), kk)
    keep = ii != jj
    pc = float((Xn[ii[keep]] * Xn[jj[keep]]).sum(1).mean())
    print(f"  top-{k} PCs removed: mean random-pair cosine {pc:+.3f}")
print("interpretation: large drop from k=0 to k=1/3 means the raw")
print("similarity was carried by a few dominant directions.")

## Block 4 — Cross-space battery (needs paired Y)

Ridge vs Procrustes on the same split answers *which kind* of difference
separates the spaces; the fitted W's singular values are its volume knobs.

In [ ]:
if Y is not None:
    idx = rng.permutation(len(X))
    ktr = int(0.75 * len(X))
    tr, te = idx[:ktr], idx[ktr:]

    # ridge, held-out
    a = 1e-2
    Wm = np.linalg.solve(X[tr].T @ X[tr] + a * np.eye(X.shape[1]),
                         X[tr].T @ Y[tr])
    P = X[te] @ Wm
    r2 = 1 - ((Y[te] - P) ** 2).sum() / ((Y[te] - Y[te].mean(0)) ** 2).sum()
    cos_w = float((l2n(P) * l2n(Y[te])).sum(1).mean())

    # shuffle control (alignment + power check)
    Ys = Y[tr][rng.permutation(ktr)]
    Ws = np.linalg.solve(X[tr].T @ X[tr] + a * np.eye(X.shape[1]),
                         X[tr].T @ Ys)
    r2s = 1 - ((Y[te] - X[te] @ Ws) ** 2).sum() /               ((Y[te] - Y[te].mean(0)) ** 2).sum()

    # procrustes (zero-pad if widths differ)
    d1, d2 = X.shape[1], Y.shape[1]
    Xp = np.pad(X, ((0, 0), (0, max(0, d2 - d1))))[:, :max(d1, d2)]
    Yp = np.pad(Y, ((0, 0), (0, max(0, d1 - d2))))[:, :max(d1, d2)]
    U, _, Vt = np.linalg.svd(Xp[tr].T @ Yp[tr])
    R = U @ Vt
    Pr = Xp[te] @ R
    r2p = 1 - ((Yp[te] - Pr) ** 2).sum() /               ((Yp[te] - Yp[te].mean(0)) ** 2).sum()
    cos_p = float((l2n(Pr) * l2n(Yp[te])).sum(1).mean())

    # CKA on the (sub)samples
    def cka(Xa, Ya):
        Xa = Xa - Xa.mean(0); Ya = Ya - Ya.mean(0)
        num = np.linalg.norm(Ya.T @ Xa) ** 2
        return float(num / (np.linalg.norm(Xa.T @ Xa) *
                            np.linalg.norm(Ya.T @ Ya)))
    sub = rng.choice(len(X), min(4000, len(X)), replace=False)
    print(f"held-out ridge   : R2 {r2:.3f}   cosine {cos_w:.3f}")
    print(f"shuffle control  : R2 {r2s:.3f}   "
          f"(honest fit must beat this by >0.2)")
    print(f"procrustes       : R2 {r2p:.3f}   cosine {cos_p:.3f}")
    print(f"  ridge >> procrustes on R2 while procrustes cosine is high")
    print(f"  -> anisotropy mismatch (volumes), not geography")
    print(f"CKA(X, Y)        : {cka(X[sub], Y[sub]):.3f}")

    sw = np.linalg.svd(Wm, compute_uv=False)
    print(f"\nvolume knobs (singular values of W):")
    print(f"  max {sw.max():.3f}  median {np.median(sw):.3f}  "
          f"min {sw.min():.3f}  spread {sw.max()/max(sw.min(),1e-9):.0f}x")
    print(f"  fraction outside [0.8, 1.25]: "
          f"{((sw < 0.8) | (sw > 1.25)).mean():.2f}")
    print("  a pure rotation would print all ~1.0")
else:
    print("no Y - skipping cross-space battery")

## Block 5 — Semantic similarity: retrieval and membership (needs paired Y)

In [ ]:
if Y is not None:
    q = l2n(X[te] @ Wm)          # adapted queries
    g = l2n(Y[te])               # gallery
    S = q @ g.T
    order = np.argsort(-S, axis=1)
    ranks = (order == np.arange(len(S))[:, None]).argmax(1)
    for k in (1, 5, 10):
        print(f"  R@{k} = {(ranks < k).mean():.3f}")
    tc = S[np.arange(len(S)), np.arange(len(S))]
    Sw = S.copy(); Sw[np.arange(len(S)), np.arange(len(S))] = -1
    gap = tc - Sw.max(1)
    print(f"  membership gap: mean {gap.mean():+.3f}  min {gap.min():+.3f}"
          f"  positive {(gap > 0).mean():.3f}")
    print("  large positive gap -> a threshold answers 'is this item")
    print("  already indexed?' across the two spaces")
else:
    print("no Y - skipping semantic battery")

## Reading the whole battery

- **Structure** (blocks 2-3) tells you the *shape* of one space: how many
  directions carry variance, whether similarity scores are inflated by a
  shared component, whether any finding survives centering.
- **Semantics** (blocks 4-5) needs a second, paired space: it tells you
  whether the two spaces carry the *same content* (CKA, ridge), what kind
  of transformation separates them (Procrustes-vs-ridge; the volume
  knobs), and whether the correspondence is good enough to *use*
  (R@K, membership gap).
- Any negative result here inherits the project's rule: verify rows/dim,
  pass the shuffle test, and check artifact sanity before believing it.